In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data_prep import build_dataset, FEATURES, FEATURES_SMALL
from src.evaluation import evaluate, bootstrap_auc
from src.models import make_model

train = build_dataset([2014, 2015, 2016])
val = build_dataset([2017])

FEATURES_BIG = FEATURES + ["has_loss"]
print("Träning:", train.shape, " Validering:", val.shape)

ImportError: cannot import name 'make_model' from 'src.models' (c:\workspace\Ml_stock_ranking\notebooks\..\src\models.py)

In [ ]:
def run(name, features, winsor=False, sector=False, label=None):
    """Tränar en modell på 2014–2016 och utvärderar den på 2017."""
    columns = features + (["Sector"] if sector else [])
    model = make_model(name, features, winsor=winsor, sector=sector)
    model.fit(train[columns], train["above_median"])
    proba = model.predict_proba(val[columns])[:, 1]

    result = evaluate(val, proba, label or name)
    low, high = bootstrap_auc(val, proba)
    result["AUC 95 %-intervall"] = f"[{low:.3f}, {high:.3f}]"
    return result

MODELS = ["Logistisk regression", "Random forest", "Gradient boosting"]

## Baseline

In [ ]:
rng = np.random.default_rng(42)
random_proba = rng.random(len(val))

# P/B-regeln: lägst P/B får högst poäng. Saknat P/B hamnar i mitten.
pb_proba = val["PB ratio"].rank(ascending=False, pct=True).fillna(0.5).to_numpy()

pd.DataFrame([
    evaluate(val, random_proba, "Slump"),
    evaluate(val, pb_proba, "Lågt P/B"),
])

## Experiment A: Vilken modell? (15 nyckeltal, ingen winsorisering, ingen sektor)

In [ ]:
pd.DataFrame([run(m, FEATURES_BIG) for m in MODELS])

## Experiment B: Hjälper winsorisering? (Henris och Marcies idé)

In [ ]:
pd.DataFrame([
    run(m, FEATURES_BIG, winsor=w, label=f"{m} | {'med' if w else 'utan'} winsor")
    for m in MODELS for w in [False, True]
])

## Experiment C: 5 eller 15 nyckeltal? (med winsorisering)

In [ ]:
pd.DataFrame([
    run(m, f, winsor=True, label=f"{m} | {n} nyckeltal")
    for m in MODELS for n, f in [(5, FEATURES_SMALL), (15, FEATURES_BIG)]
])

## Experiment D: Hjälper sektor? (15 nyckeltal, med winsorisering)

In [ ]:
pd.DataFrame([
    run(m, FEATURES_BIG, winsor=True, sector=s, label=f"{m} | {'med' if s else 'utan'} sektor")
    for m in MODELS for s in [False, True]
])